# Variance-Weighted Demosaic Spot Detection Validation

This notebook compares spot detection performance between:
1. **Standard demosaicing**: `bayer_demosaic_stack` function
2. **Variance-aware demosaicing**: `variance_aware_malvar_demosaic` function

We generate 1,000 synthetic images with known ground truth puncta locations and evaluate:
- **Type I Error** (False Positives): How many "junk spots" are detected?
- **Type II Error** (False Negatives): How many real spots are missed?
- **True Positives**: How many real spots are correctly detected?

A spot is considered "detected" if it's within 2 pixels of a ground truth location.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import pandas as pd
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from typing import Tuple, Dict, List
from scipy.spatial.distance import cdist
import gc
from dataclasses import dataclass

import sys

sys.path.append("..")

# Modern import pattern - unified simulation method with strategy pattern
from src import Multicolour_Simulation_Functions
from src.Multicolour_Simulation_Functions import FittingStrategy, SimulationConfig

# Additional required components
from src import PlottingFunctions
from src import SpectralFunctions
from src import MaskFunctions
from src import HelperFunctions
from src import SpotDetectionFunctions

# Create main simulation instance
MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

# Access integrated components through MSF
IO = MSF.io
I_AF = MSF.image_analysis
sCMOS = MSF.scmos
PSF = MSF.psf

# Create instances of non-integrated components
plotter = PlottingFunctions.Plotter()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
H_F = HelperFunctions.Helper_Functions()
SD_F = SpotDetectionFunctions.SpotDetection_Functions()

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20250910_090825.log


In [2]:
# Import variance-aware demosaicing functions from original notebook
from scipy import ndimage
from typing import Literal, cast
from numpy.typing import NDArray
from numpy import (
    array,
    float32,
    float64,
    ones,
    zeros,
    logical_and,
    logical_or,
    transpose,
    where,
)

# Type aliases
NDArrayFloat = NDArray[np.float64]
ArrayLike = NDArray | list | tuple


def as_float_array(array: ArrayLike) -> NDArrayFloat:
    """Convert input to float64 array."""
    return np.asarray(array, dtype=float64)


def masks_CFA_Bayer(
    shape: tuple[int, int], pattern: str = "RGGB"
) -> tuple[NDArrayFloat, NDArrayFloat, NDArrayFloat]:
    """Return the Bayer CFA red, green and blue masks for given pattern."""
    pattern = pattern.upper()

    r = zeros(shape)
    g = zeros(shape)
    b = zeros(shape)

    # Create masks based on pattern
    if pattern == "RGGB":
        r[0::2, 0::2] = 1
        g[0::2, 1::2] = 1
        g[1::2, 0::2] = 1
        b[1::2, 1::2] = 1
    elif pattern == "BGGR":
        b[0::2, 0::2] = 1
        g[0::2, 1::2] = 1
        g[1::2, 0::2] = 1
        r[1::2, 1::2] = 1
    elif pattern == "GRBG":
        g[0::2, 0::2] = 1
        r[0::2, 1::2] = 1
        b[1::2, 0::2] = 1
        g[1::2, 1::2] = 1
    elif pattern == "GBRG":
        g[0::2, 0::2] = 1
        b[0::2, 1::2] = 1
        r[1::2, 0::2] = 1
        g[1::2, 1::2] = 1
    else:
        raise ValueError("Invalid pattern. Choose from: RGGB, BGGR, GRBG, GBRG")

    return r, g, b


def weighted_convolve(
    image: NDArrayFloat, kernel: NDArrayFloat, weights: NDArrayFloat
) -> NDArrayFloat:
    """Perform a weighted convolution where each pixel's contribution is scaled by its weight."""
    kernel_height, kernel_width = kernel.shape
    pad_height = kernel_height // 2
    pad_width = kernel_width // 2

    # Pad the image and weights
    image_padded = np.pad(
        image, ((pad_height, pad_height), (pad_width, pad_width)), mode="reflect"
    )
    weights_padded = np.pad(
        weights, ((pad_height, pad_height), (pad_width, pad_width)), mode="reflect"
    )

    result = np.zeros_like(image)

    # Precompute the sum of the kernel for normalization
    kernel_sum = np.sum(kernel)

    # Perform weighted convolution
    for i in range(pad_height, image.shape[0] + pad_height):
        for j in range(pad_width, image.shape[1] + pad_width):
            # Extract patches
            image_patch = image_padded[
                i - pad_height : i + pad_height + 1, j - pad_width : j + pad_width + 1
            ]
            weights_patch = weights_padded[
                i - pad_height : i + pad_height + 1, j - pad_width : j + pad_width + 1
            ]

            # Calculate weighted sum
            weighted_kernel = kernel * weights_patch
            weighted_sum = np.sum(image_patch * weighted_kernel)

            # Normalize by the sum of the weighted kernel
            weight_sum = np.sum(weighted_kernel)
            # Avoid division by zero - fall back to standard convolution
            if abs(weight_sum) > 1e-12:
                result[i - pad_height, j - pad_width] = weighted_sum / weight_sum
            else:
                # Fallback: use standard convolution
                result[i - pad_height, j - pad_width] = (
                    np.sum(image_patch * kernel) / kernel_sum
                )

    return result


def tstack(a: ArrayLike) -> NDArrayFloat:
    """Stack the given array of images along the last dimension."""
    a = as_float_array(a)
    return np.concatenate([x[..., None] for x in a], axis=-1)


def variance_aware_malvar_demosaic(
    CFA: NDArrayFloat,
    variance_map: NDArrayFloat,
    pattern: str = "RGGB",
    offset_map: NDArrayFloat | None = None,
    gain: float = 1.0,
) -> NDArrayFloat:
    """
    Alternative approach to variance-aware Malvar demosaicing.
    Applies variance weighting to the input before standard Malvar demosaicing.
    """
    CFA = np.squeeze(as_float_array(CFA))
    variance_map = np.squeeze(as_float_array(variance_map))

    # Apply offset correction if provided
    if offset_map is not None:
        offset_map = np.squeeze(as_float_array(offset_map))
        CFA = CFA - offset_map

    # Convert from ADU to photoelectrons
    CFA_pe = CFA / gain
    variance_pe = variance_map / (gain**2)

    # Calculate weights from variance (inverse variance weighting)
    weights = 1.0 / (variance_pe + 1e-12)

    # Apply variance weighting to the CFA data
    weighted_CFA = CFA_pe * weights

    # Normalize by the average weight to maintain overall intensity
    avg_weight = np.mean(weights)
    weighted_CFA = weighted_CFA / avg_weight

    # Apply standard Malvar demosaicing to the weighted CFA
    result, _ = sCMOS.bayer_demosaic_stack(weighted_CFA)

    return result

## Simulation Configuration

In [3]:
@dataclass
class SpotDetectionConfig:
    """Configuration for spot detection validation experiment."""

    n_images: int = 100  # Number of test images to generate per parameter combination
    image_size: int = 200  # Size of each test image (pixels)
    background_photons: int = 40  # Background photons per pixel
    pixel_size: float = 69.0  # Camera pixel size (nm)
    NA: float = 1.49  # Numerical aperture
    detection_threshold: float = 2.0  # Distance threshold for "detected" spots (pixels)
    pfa_values: List[float] = (
        None  # List of PFA values to scan (if None, uses single pfa)
    )
    pfa: float = 1e-3  # Single PFA value (used if pfa_values is None)
    photon_values: List[int] = None  # List of photon counts to scan (if None, uses single n_photons)
    n_photons: int = 1000  # Single photon value (used if photon_values is None)
    dye: str = "ATTO 565"  # Dye to use for all experiments
    sigma: float = 0.0  # Sigma parameter for spot detection
    fraction_true: float = 0.0  # Fraction true parameter for spot detection
    save_images: bool = False  # Whether to save example images
    results_folder: str = (
        "/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection"  # Folder to save results
    )

    def __post_init__(self):
        """Set default values if not provided."""
        if self.pfa_values is None:
            self.pfa_values = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

        if self.photon_values is None:
            # Generate 100 logarithmically spaced photon values from 500 to 100,000
            self.photon_values = np.logspace(np.log10(500), np.log10(100000), 100).astype(int).tolist()

        # Create results folder if it doesn't exist
        os.makedirs(self.results_folder, exist_ok=True)

    @property
    def total_parameter_combinations(self) -> int:
        """Total number of parameter combinations to test."""
        return len(self.pfa_values) * len(self.photon_values)

    @property
    def total_images(self) -> int:
        """Total number of images that will be generated."""
        return self.n_images * self.total_parameter_combinations


@dataclass
class DetectionResults:
    """Results from spot detection analysis."""

    true_positives: int
    false_positives: int
    false_negatives: int
    total_ground_truth: int
    total_detected: int

    @property
    def sensitivity(self) -> float:
        """True positive rate (1 - Type II error)."""
        return (
            self.true_positives / self.total_ground_truth
            if self.total_ground_truth > 0
            else 0.0
        )

    @property
    def precision(self) -> float:
        """Positive predictive value (1 - Type I error rate)."""
        return (
            self.true_positives / self.total_detected
            if self.total_detected > 0
            else 0.0
        )

    @property
    def f1_score(self) -> float:
        """Harmonic mean of precision and recall."""
        if self.precision + self.sensitivity == 0:
            return 0.0
        return (
            2
            * (self.precision * self.sensitivity)
            / (self.precision + self.sensitivity)
        )

    @property
    def type_i_error_rate(self) -> float:
        """False positive rate."""
        return (
            self.false_positives / self.total_detected
            if self.total_detected > 0
            else 0.0
        )

    @property
    def type_ii_error_rate(self) -> float:
        """False negative rate."""
        return (
            self.false_negatives / self.total_ground_truth
            if self.total_ground_truth > 0
            else 0.0
        )


# Create configuration with photon scanning
config = SpotDetectionConfig(
    n_images=1000,  # Reduced since we're scanning many photon values
    pfa_values=[1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    dye="ATTO 550",  # Single dye for all experiments
    results_folder="/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection",
)

print(f"Configuration: {config.n_images} images per parameter combination")
print(f"Image size: {config.image_size}x{config.image_size} pixels")
print(f"Detection threshold: {config.detection_threshold} pixels")
print(f"PFA values to test: {config.pfa_values}")
print(f"Photon values to test: {len(config.photon_values)} values from {min(config.photon_values)} to {max(config.photon_values)}")
print(f"Dye: {config.dye}")
print(f"Total parameter combinations: {config.total_parameter_combinations}")
print(f"Total images to generate: {config.total_images}")
print(f"Background: {config.background_photons} photons/pixel")
print(f"Results will be saved to: {config.results_folder}")

Configuration: 1000 images per parameter combination
Image size: 200x200 pixels
Detection threshold: 2.0 pixels
PFA values to test: [1e-05, 5e-05, 0.0001, 0.0005, 0.001, 0.005, 0.01]
Photon values to test: 100 values from 499 to 100000
Dye: ATTO 550
Total parameter combinations: 700
Total images to generate: 700000
Background: 40 photons/pixel
Results will be saved to: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection


## Load Camera Calibration and Setup

In [4]:
# Load camera calibration data
data_folder = "../Camera_Calibrations/Ximea_Camera/"
gain_full = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset_full = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance_full = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise_full = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe_full = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

# Get pixel quantum yields and create masks
R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])
masks = M_F.get_masks(size_x=config.image_size, size_y=config.image_size)

# Set up filters for spectral analysis
notch_filter = "semrock-nf03-405-488-561-635e"
dichroic_mirror = "semrock-di03-r405-488-561-635-t1-25x36"
shortpass_filter = "semrock-bsp01-785r"
filters = [notch_filter, dichroic_mirror, shortpass_filter]

# Set up smoothing function
import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"Camera calibration loaded: {gain_full.shape}")
print(f"Will crop to {config.image_size}x{config.image_size} regions")

Camera calibration loaded: (1544, 2064)
Will crop to 200x200 regions


## Ground Truth Puncta Generation

In [5]:
def generate_puncta_grid(
    image_size: int, pixel_size: float, spacing_factor: float = 0.05
) -> Tuple[np.ndarray, int]:
    """
    Generate a grid of puncta positions with some randomization.

    Args:
        image_size: Size of image in pixels
        pixel_size: Size of each pixel in nm
        spacing_factor: Fraction of image size between puncta (0.05 = 5%)

    Returns:
        positions: Array of [x, y] positions in nm
        n_puncta: Number of puncta generated
    """
    # Create grid positions with slight randomization
    x = np.linspace(spacing_factor, 1 - spacing_factor, int(1 / spacing_factor) - 1)
    X, Y = np.meshgrid(x, x)
    X = X.ravel()
    Y = Y.ravel()

    # Add small random offset to each position (±10% of spacing)
    spacing = spacing_factor * image_size
    offset_std = spacing * 0.1
    X += np.random.normal(0, offset_std / image_size, len(X))
    Y += np.random.normal(0, offset_std / image_size, len(Y))

    # Clip to image boundaries
    X = np.clip(X, 0.01, 0.99)
    Y = np.clip(Y, 0.01, 0.99)

    # Convert to nm coordinates
    positions = np.array([X * (image_size * pixel_size), Y * (image_size * pixel_size)])

    return positions, len(X)


def generate_camera_parameters(full_calibration: dict, image_size: int) -> dict:
    """
    Generate camera parameters for a random crop of the full calibration.

    Args:
        full_calibration: Dictionary with full camera calibration arrays
        image_size: Size of the crop in pixels

    Returns:
        Camera parameters dictionary for the cropped region
    """
    # Random crop location
    max_x = full_calibration["gain"].shape[0] - image_size
    max_y = full_calibration["gain"].shape[1] - image_size

    start_x = np.random.randint(0, max_x)
    start_y = np.random.randint(0, max_y)

    # Crop all calibration arrays
    camera_parameters = {
        "gain": full_calibration["gain"][
            start_x : start_x + image_size, start_y : start_y + image_size
        ],
        "variance": full_calibration["variance"][
            start_x : start_x + image_size, start_y : start_y + image_size
        ],
        "readnoise": full_calibration["readnoise"][
            start_x : start_x + image_size, start_y : start_y + image_size
        ],
        "offset": full_calibration["offset"][
            start_x : start_x + image_size, start_y : start_y + image_size
        ],
        "rqe": full_calibration["rqe"][
            start_x : start_x + image_size, start_y : start_y + image_size
        ],
        "pixel_QYs": pixel_QYs,
        "pixel_order": ["B", "G", "R"],
        "pixel_order_indices": [0, 1, 2],
        "masks": masks,
    }

    return camera_parameters


# Store full calibration in dictionary for easy access
full_calibration = {
    "gain": gain_full,
    "offset": offset_full,
    "variance": variance_full,
    "readnoise": readnoise_full,
    "rqe": rqe_full,
}

# Test puncta generation
test_positions, n_test_puncta = generate_puncta_grid(
    config.image_size, config.pixel_size
)
print(f"Generated {n_test_puncta} puncta in test grid")
print(
    f"Position range: X=[{test_positions[0].min():.0f}, {test_positions[0].max():.0f}] nm"
)
print(
    f"Position range: Y=[{test_positions[1].min():.0f}, {test_positions[1].max():.0f}] nm"
)

Generated 361 puncta in test grid
Position range: X=[553, 13276] nm
Position range: Y=[529, 13212] nm


## Spot Detection Validation Functions

In [6]:
def analyze_detections(
    ground_truth: np.ndarray, detected: np.ndarray, threshold_pixels: float
) -> DetectionResults:
    """
    Analyze detection performance by comparing ground truth to detected spots.

    Args:
        ground_truth: Array of ground truth positions (pixel coordinates) shape (N, 2)
        detected: Array of detected positions (pixel coordinates) shape (M, 2)
        threshold_pixels: Distance threshold for considering a detection "correct"

    Returns:
        DetectionResults object with performance metrics
    """
    if len(detected) == 0:
        return DetectionResults(
            true_positives=0,
            false_positives=0,
            false_negatives=len(ground_truth),
            total_ground_truth=len(ground_truth),
            total_detected=0,
        )

    if len(ground_truth) == 0:
        return DetectionResults(
            true_positives=0,
            false_positives=len(detected),
            false_negatives=0,
            total_ground_truth=0,
            total_detected=len(detected),
        )

    # Calculate distances between all detected and ground truth spots
    distances = cdist(detected, ground_truth)

    # Find matches: detected spots within threshold of ground truth
    matches = distances <= threshold_pixels

    # For each ground truth spot, find if any detection is close enough
    gt_matched = np.any(matches, axis=0)
    true_positives = np.sum(gt_matched)
    false_negatives = len(ground_truth) - true_positives

    # For each detection, check if it matches any ground truth
    det_matched = np.any(matches, axis=1)
    detection_true_positives = np.sum(det_matched)
    false_positives = len(detected) - detection_true_positives

    return DetectionResults(
        true_positives=true_positives,
        false_positives=false_positives,
        false_negatives=false_negatives,
        total_ground_truth=len(ground_truth),
        total_detected=len(detected),
    )


def convert_positions_to_pixels(
    positions_nm: np.ndarray, pixel_size: float
) -> np.ndarray:
    """
    Convert positions from nm to pixel coordinates.

    Args:
        positions_nm: Positions in nm, shape (2, N) as [x_coords, y_coords]
        pixel_size: Size of each pixel in nm

    Returns:
        positions_pixels: Positions in pixels, shape (N, 2) as [[x1,y1], [x2,y2], ...]
    """
    positions_pixels = positions_nm.T / pixel_size  # Convert to pixels and transpose
    return positions_pixels


def detect_spots_three_methods(
    image_data: dict, pfa: float, sigma: float = 0.0, fraction_true: float = 0.0
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Detect spots using three different methods:
    1. Standard demosaicing → grayscale → spot detection
    2. Variance-aware demosaicing → grayscale → spot detection
    3. Raw Bayer → grayscale → spot detection (no demosaicing)

    Args:
        image_data: Dictionary containing 'bayer', 'camera_params' keys
        pfa: Probability of false alarm for detection
        sigma: Sigma parameter for spot detection
        fraction_true: Fraction true parameter for spot detection

    Returns:
        detected_standard: Spots detected using standard demosaicing method
        detected_variance: Spots detected using variance-aware demosaicing method
        detected_raw: Spots detected using raw Bayer → grayscale method
    """
    bayer_image = image_data["bayer"]
    camera_params = image_data["camera_params"]

    # Method 1: Standard demosaicing
    photoelectrons = IO.convert_to_photoelectrons(
        bayer_image,
        gain_map=camera_params["gain"],
        offset_map=camera_params["offset"],
        rqe=camera_params["rqe"],
    )
    demosaic_standard, _ = sCMOS.bayer_demosaic_stack(photoelectrons)
    grayscale_standard = np.sum(demosaic_standard, axis=-1)

    detected_standard = SD_F.detect_puncta_in_image(
        image=grayscale_standard,
        variance=camera_params["variance"],
        pfa=pfa,
        sigma=sigma,
        fraction_true=fraction_true,
    )

    # Method 2: Variance-aware demosaicing
    demosaic_variance = variance_aware_malvar_demosaic(
        bayer_image,
        variance_map=camera_params["variance"],
        gain=camera_params["gain"],
        offset_map=camera_params["offset"],
    )
    grayscale_variance = np.sum(demosaic_variance, axis=-1)

    detected_variance = SD_F.detect_puncta_in_image(
        image=grayscale_variance, pfa=pfa, sigma=sigma, fraction_true=fraction_true
    )

    # Method 3: Raw Bayer → grayscale (no demosaicing)
    # Convert raw Bayer to photoelectrons first
    photoelectrons_raw = IO.convert_to_photoelectrons(
        bayer_image,
        gain_map=camera_params["gain"],
        offset_map=camera_params["offset"],
        rqe=camera_params["rqe"],
    )

    # Convert directly to grayscale by simple averaging (no demosaicing interpolation)
    grayscale_raw = photoelectrons_raw.astype(np.float32)

    detected_raw = SD_F.detect_puncta_in_image(
        image=grayscale_raw,
        variance=camera_params["variance"],
        pfa=pfa,
        sigma=sigma,
        fraction_true=fraction_true,
    )

    return detected_standard, detected_variance, detected_raw


# Test the analysis functions
print("Testing detection analysis functions...")

# Create test data
test_gt = np.array([[10, 10], [20, 20], [30, 30]])
test_det = np.array(
    [[10.5, 10.2], [19.8, 20.1], [50, 50]]
)  # First two close, third is false positive

test_results = analyze_detections(test_gt, test_det, 2.0)
print(
    f"Test results: TP={test_results.true_positives}, FP={test_results.false_positives}, FN={test_results.false_negatives}"
)
print(
    f"Sensitivity: {test_results.sensitivity:.2f}, Precision: {test_results.precision:.2f}, F1: {test_results.f1_score:.2f}"
)

print("All detection analysis functions defined successfully!")

Testing detection analysis functions...
Test results: TP=2, FP=1, FN=1
Sensitivity: 0.67, Precision: 0.67, F1: 0.67
All detection analysis functions defined successfully!


## Main Validation Experiment

In [7]:
def run_parameter_grid_scan(config: SpotDetectionConfig) -> pd.DataFrame:
    """
    Run validation experiment across multiple PFA values and photon counts (grid search).
    Tests three methods: Standard, Variance-Aware, and Raw Bayer.

    Args:
        config: Experimental configuration with pfa_values and photon_values to scan

    Returns:
        results_df: DataFrame with all results across parameter combinations
    """
    print(f"Starting parameter grid scan:")
    print(f"  - {len(config.pfa_values)} PFA values: {config.pfa_values}")
    print(f"  - {len(config.photon_values)} photon values: {min(config.photon_values)} to {max(config.photon_values)}")
    print(f"  - 3 methods: Standard, Variance-Aware, Raw Bayer")
    print(f"  - {config.total_parameter_combinations} total parameter combinations")
    print(f"  - {config.total_images} total images to generate")

    all_results = []

    # Track overall progress
    combo_count = 0
    total_combos = config.total_parameter_combinations
    overall_start_time = time.time()

    for photon_idx, n_photons in enumerate(config.photon_values):
        for pfa_idx, pfa in enumerate(config.pfa_values):
            combo_count += 1

            print(f"\n{'='*80}")
            print(f"PARAMETER COMBINATION {combo_count}/{total_combos}")
            print(f"Photons: {n_photons} | PFA: {pfa:.2e}")
            print(f"{'='*80}")

            # Calculate ETA
            if combo_count > 1:
                elapsed = time.time() - overall_start_time
                rate = (combo_count - 1) / elapsed
                eta_total = (total_combos - combo_count + 1) / rate if rate > 0 else 0
                print(
                    f"Overall progress: {100*combo_count/total_combos:.1f}% | ETA: {eta_total/60:.1f} min"
                )

            # Run experiment for this parameter combination
            results_std, results_var, results_raw = (
                run_validation_experiment_single_combination(config, pfa, n_photons)
            )

            # Convert results to dataframe rows
            for img_idx in range(len(results_std)):
                # Standard method result
                std_result = results_std[img_idx]
                all_results.append(
                    {
                        "pfa": pfa,
                        "n_photons": n_photons,
                        "image_id": img_idx,
                        "method": "Standard",
                        "sensitivity": std_result.sensitivity,
                        "precision": std_result.precision,
                        "f1_score": std_result.f1_score,
                        "type_i_error": std_result.type_i_error_rate,
                        "type_ii_error": std_result.type_ii_error_rate,
                        "true_positives": std_result.true_positives,
                        "false_positives": std_result.false_positives,
                        "false_negatives": std_result.false_negatives,
                        "total_ground_truth": std_result.total_ground_truth,
                        "total_detected": std_result.total_detected,
                    }
                )

                # Variance-aware method result
                var_result = results_var[img_idx]
                all_results.append(
                    {
                        "pfa": pfa,
                        "n_photons": n_photons,
                        "image_id": img_idx,
                        "method": "Variance-Aware",
                        "sensitivity": var_result.sensitivity,
                        "precision": var_result.precision,
                        "f1_score": var_result.f1_score,
                        "type_i_error": var_result.type_i_error_rate,
                        "type_ii_error": var_result.type_ii_error_rate,
                        "true_positives": var_result.true_positives,
                        "false_positives": var_result.false_positives,
                        "false_negatives": var_result.false_negatives,
                        "total_ground_truth": var_result.total_ground_truth,
                        "total_detected": var_result.total_detected,
                    }
                )

                # Raw Bayer method result
                raw_result = results_raw[img_idx]
                all_results.append(
                    {
                        "pfa": pfa,
                        "n_photons": n_photons,
                        "image_id": img_idx,
                        "method": "Raw Bayer",
                        "sensitivity": raw_result.sensitivity,
                        "precision": raw_result.precision,
                        "f1_score": raw_result.f1_score,
                        "type_i_error": raw_result.type_i_error_rate,
                        "type_ii_error": raw_result.type_ii_error_rate,
                        "true_positives": raw_result.true_positives,
                        "false_positives": raw_result.false_positives,
                        "false_negatives": raw_result.false_negatives,
                        "total_ground_truth": raw_result.total_ground_truth,
                        "total_detected": raw_result.total_detected,
                    }
                )

    total_time = time.time() - overall_start_time
    print(f"\n{'='*80}")
    print(f"PARAMETER GRID SCAN COMPLETE!")
    print(f"Total time: {total_time/60:.1f} minutes")
    print(f"Average time per combination: {total_time/total_combos:.1f} seconds")
    print(f"{'='*80}")

    results_df = pd.DataFrame(all_results)
    return results_df


def run_validation_experiment_single_combination(
    config: SpotDetectionConfig, pfa: float, n_photons: int
) -> Tuple[List[DetectionResults], List[DetectionResults], List[DetectionResults]]:
    """
    Run the validation experiment for a single PFA and photon count combination.
    Tests all three methods: Standard, Variance-Aware, and Raw Bayer.

    Args:
        config: Experimental configuration
        pfa: Probability of false alarm value to test
        n_photons: Number of photons per spot to test

    Returns:
        results_standard: List of results for standard method
        results_variance: List of results for variance-aware method
        results_raw: List of results for raw Bayer method
    """

    print(
        f"Starting validation with {config.n_images} images (PFA = {pfa:.2e}, Photons = {n_photons})..."
    )
    print(
        f"Each image: {config.image_size}x{config.image_size} pixels, {n_photons} photons per spot"
    )
    print(f"Testing 3 methods: Standard, Variance-Aware, Raw Bayer")

    # Get dye properties - use single dye for all experiments
    try:
        average_emission_wavelengths, dye_pixel_efficiency = (
            S_F.get_pixel_fractions_dye_and_filters(config.dye, filters, wavelength, pixel_QYs)
        )
    except Exception as e:
        print(f"Error getting dye properties for {config.dye}: {e}")
        print(f"Available dyes might be limited - check SpectralFunctions database")
        # Return empty results for this combination
        empty_result = DetectionResults(0, 0, 0, 0, 0)
        return (
            [empty_result] * config.n_images,
            [empty_result] * config.n_images,
            [empty_result] * config.n_images,
        )

    # Storage for results
    results_standard = []
    results_variance = []
    results_raw = []

    # Progress tracking
    start_time = time.time()

    for i in range(config.n_images):
        # Progress reporting every 25 images
        if i % 25 == 0 or i == config.n_images - 1:
            current_time = time.time()
            elapsed = current_time - start_time
            if i > 0:
                rate = i / elapsed
                eta = (config.n_images - i) / rate
                print(
                    f"  Progress: {i+1:3d}/{config.n_images} ({100*(i+1)/config.n_images:5.1f}%) - {rate:.1f} img/s - ETA: {eta/60:.1f}min"
                )
            else:
                print(
                    f"  Progress: {i+1:3d}/{config.n_images} ({100*(i+1)/config.n_images:5.1f}%)"
                )

        try:
            # Generate ground truth puncta positions
            positions_nm, n_puncta = generate_puncta_grid(
                config.image_size, config.pixel_size
            )
            ground_truth_pixels = convert_positions_to_pixels(
                positions_nm, config.pixel_size
            )

            # Generate random camera parameters for this image
            camera_params = generate_camera_parameters(
                full_calibration, config.image_size
            )

            # Set up photon and position data for image generation with current photon count
            photon_dict = {config.dye: np.full(1, n_photons)}
            x0y0 = {config.dye: np.zeros([1, 2, len(positions_nm[0])])}
            x0y0[config.dye][0, :, :] = positions_nm

            # Generate synthetic camera image
            bayer_image, smoothed_image, _ = MSF.gen_camera_image_stack(
                camera_params,
                wavelength,
                average_emission_wavelengths,
                dye_pixel_efficiency,
                photon_dict,
                x0y0,
                smoothing_function=smoothing_function,
                background_photons=config.background_photons,
                NA=config.NA,
                pixel_size=config.pixel_size,
                return_normal_image=False,
            )

            # Prepare image data for detection
            image_data = {
                "bayer": np.squeeze(bayer_image),
                "camera_params": camera_params,
            }

            # Detect spots using all three methods
            detected_standard, detected_variance, detected_raw = (
                detect_spots_three_methods(
                    image_data, pfa, config.sigma, config.fraction_true
                )
            )

            # Analyze results for all three methods
            result_standard = analyze_detections(
                ground_truth_pixels, detected_standard, config.detection_threshold
            )
            result_variance = analyze_detections(
                ground_truth_pixels, detected_variance, config.detection_threshold
            )
            result_raw = analyze_detections(
                ground_truth_pixels, detected_raw, config.detection_threshold
            )

            results_standard.append(result_standard)
            results_variance.append(result_variance)
            results_raw.append(result_raw)

            # Force garbage collection every 25 images to manage memory
            if i % 25 == 24:
                gc.collect()

        except Exception as e:
            print(f"  Error processing image {i+1}: {e}")
            # Add dummy results to maintain array length
            dummy_result = DetectionResults(0, 0, 0, 0, 0)
            results_standard.append(dummy_result)
            results_variance.append(dummy_result)
            results_raw.append(dummy_result)
            continue

    total_time = time.time() - start_time
    print(
        f"  Validation complete! Processed {config.n_images} images in {total_time/60:.1f} minutes"
    )
    print(f"  Average rate: {config.n_images/total_time:.1f} images/second")

    return results_standard, results_variance, results_raw


print("=" * 60)
print("STARTING PARAMETER GRID SCAN EXPERIMENT")
print("=" * 60)

# Run parameter grid scan
all_results_df = run_parameter_grid_scan(config)

STARTING PARAMETER GRID SCAN EXPERIMENT
Starting parameter grid scan:
  - 7 PFA values: [1e-05, 5e-05, 0.0001, 0.0005, 0.001, 0.005, 0.01]
  - 100 photon values: 499 to 100000
  - 3 methods: Standard, Variance-Aware, Raw Bayer
  - 700 total parameter combinations
  - 700000 total images to generate

PARAMETER COMBINATION 1/700
Photons: 499 | PFA: 1.00e-05
Starting validation with 1000 images (PFA = 1.00e-05, Photons = 499)...
Each image: 200x200 pixels, 499 photons per spot
Testing 3 methods: Standard, Variance-Aware, Raw Bayer
  Progress:   1/1000 (  0.1%)
  Progress:  26/1000 (  2.6%) - 6.0 img/s - ETA: 2.7min
  Progress:  51/1000 (  5.1%) - 8.0 img/s - ETA: 2.0min
  Progress:  76/1000 (  7.6%) - 9.1 img/s - ETA: 1.7min
  Progress: 101/1000 ( 10.1%) - 9.7 img/s - ETA: 1.5min
  Progress: 126/1000 ( 12.6%) - 10.1 img/s - ETA: 1.4min
  Progress: 151/1000 ( 15.1%) - 10.4 img/s - ETA: 1.4min
  Progress: 176/1000 ( 17.6%) - 10.6 img/s - ETA: 1.3min
  Progress: 201/1000 ( 20.1%) - 10.7 img/

In [8]:
# Save comprehensive results to specified folder
timestamp = time.strftime("%Y%m%d_%H%M%S")
results_filename = os.path.join(
    config.results_folder, f"photon_scan_results_{timestamp}.csv"
)
all_results_df.to_csv(results_filename, index=False)

# Save configuration
config_df = pd.DataFrame(
    [
        {
            "n_images": config.n_images,
            "image_size": config.image_size,
            "background_photons": config.background_photons,
            "pixel_size": config.pixel_size,
            "NA": config.NA,
            "detection_threshold": config.detection_threshold,
            "pfa_values": str(config.pfa_values),  # Convert list to string for CSV
            "photon_values_min": min(config.photon_values),
            "photon_values_max": max(config.photon_values),
            "photon_values_count": len(config.photon_values),
            "dye": config.dye,
            "sigma": config.sigma,
            "fraction_true": config.fraction_true,
            "total_parameter_combinations": config.total_parameter_combinations,
            "total_images": config.total_images,
            "methods_tested": "Standard, Variance-Aware, Raw Bayer",
            "results_folder": config.results_folder,
            "timestamp": timestamp,
        }
    ]
)

config_filename = os.path.join(
    config.results_folder, f"photon_scan_config_{timestamp}.csv"
)
config_df.to_csv(config_filename, index=False)

# Save summary statistics grouped by all parameters
summary_stats = all_results_df.groupby(["pfa", "n_photons", "method"]).agg(
    {
        "sensitivity": ["mean", "std", "count"],
        "precision": ["mean", "std", "count"],
        "f1_score": ["mean", "std", "count"],
        "type_i_error": ["mean", "std", "count"],
        "type_ii_error": ["mean", "std", "count"],
        "true_positives": "sum",
        "false_positives": "sum",
        "false_negatives": "sum",
    }
)

summary_filename = os.path.join(
    config.results_folder, f"photon_scan_summary_{timestamp}.csv"
)
summary_stats.to_csv(summary_filename)

# Save optimal parameter combinations for all three methods
optimal_params = (
    all_results_df.groupby(["pfa", "n_photons", "method"])["f1_score"].mean().reset_index()
)
optimal_params_pivot = optimal_params.pivot(
    index=["pfa", "n_photons"], columns="method", values="f1_score"
)

# Find best combinations for each method
available_methods = [
    col
    for col in ["Standard", "Variance-Aware", "Raw Bayer"]
    if col in optimal_params_pivot.columns
]

if len(available_methods) >= 2:
    optimal_combinations = []

    for method in available_methods:
        best_idx = optimal_params_pivot[method].idxmax()
        optimal_combinations.append(
            {
                "method": method,
                "optimal_pfa": best_idx[0],
                "optimal_n_photons": best_idx[1],
                "f1_score": optimal_params_pivot.loc[best_idx, method],
                "timestamp": timestamp,
            }
        )

    optimal_df = pd.DataFrame(optimal_combinations)
    optimal_filename = os.path.join(
        config.results_folder, f"photon_scan_optimal_parameters_{timestamp}.csv"
    )
    optimal_df.to_csv(optimal_filename, index=False)

    print(f"  Optimal parameters: {optimal_filename}")

print(f"\nResults saved to:")
print(f"  Main results: {results_filename}")
print(f"  Configuration: {config_filename}")
print(f"  Summary statistics: {summary_filename}")
print(f"  All files saved in folder: {config.results_folder}")

# Display final summary
print(f"\n" + "=" * 80)
print("PHOTON SCAN PARAMETER GRID SCAN SUMMARY")
print("=" * 80)

print(f"Experiment completed successfully!")
print(f"  - Generated and analyzed {config.total_images} synthetic images total")
print(f"  - {len(config.pfa_values)} PFA values tested: {config.pfa_values}")
print(f"  - {len(config.photon_values)} photon values tested: {min(config.photon_values)} to {max(config.photon_values)}")
print(f"  - Dye used: {config.dye}")
print(f"  - 3 methods tested: Standard, Variance-Aware, Raw Bayer")
print(f"  - {config.total_parameter_combinations} parameter combinations tested")
print(
    f"  - Each image: {config.image_size}x{config.image_size} pixels with ~{generate_puncta_grid(config.image_size, config.pixel_size)[1]} puncta"
)
print(f"  - Detection threshold: {config.detection_threshold} pixels")
print(f"  - Background: {config.background_photons} photons/pixel")

# Overall performance comparison across all parameter combinations
method_f1_scores = all_results_df.groupby("method")["f1_score"].mean()
print(f"\nOverall performance (averaged across all parameter combinations):")
for method, f1_score in method_f1_scores.items():
    print(f"  - {method:15}: F1 = {f1_score:.3f}")

# Find and display best parameter combinations
if len(available_methods) >= 2:
    print(f"\nOptimal parameter combinations:")
    for combo in optimal_combinations:
        print(
            f"  - {combo['method']:15}: PFA={combo['optimal_pfa']:.2e}, Photons={combo['optimal_n_photons']:5d} (F1 = {combo['f1_score']:.3f})"
        )

# Performance ranking
best_method_overall = method_f1_scores.idxmax()
method_ranking = method_f1_scores.sort_values(ascending=False)

print(f"\nOverall method ranking:")
for rank, (method, score) in enumerate(method_ranking.items(), 1):
    print(f"  {rank}. {method:15}: F1 = {score:.3f}")

# Performance assessment
variance_f1 = method_f1_scores.get("Variance-Aware", 0)
standard_f1 = method_f1_scores.get("Standard", 0)
raw_f1 = method_f1_scores.get("Raw Bayer", 0)

variance_vs_standard = variance_f1 - standard_f1
variance_vs_raw = variance_f1 - raw_f1
standard_vs_raw = standard_f1 - raw_f1

print(f"\nMethod comparisons:")
print(f"  - Variance-Aware vs Standard: {variance_vs_standard:+.3f}")
print(f"  - Variance-Aware vs Raw Bayer: {variance_vs_raw:+.3f}")
print(f"  - Standard vs Raw Bayer: {standard_vs_raw:+.3f}")

if best_method_overall == "Variance-Aware" and variance_vs_standard > 0.01:
    print(
        f"\n🎉 Variance-aware demosaicing shows significant improvement over both Standard and Raw Bayer methods!"
    )
elif best_method_overall == "Variance-Aware" and variance_vs_standard > 0.001:
    print(
        f"\n✅ Variance-aware demosaicing shows modest improvement over other methods."
    )
elif best_method_overall == "Raw Bayer":
    print(
        f"\n📊 Surprisingly, Raw Bayer performs best - demosaicing may be adding noise!"
    )
else:
    print(f"\n📊 Standard demosaicing performs best among the three methods.")

print(
    f"\nAll results and summary statistics saved in: {config.results_folder}"
)
print("=" * 80)

  Optimal parameters: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection/photon_scan_optimal_parameters_20250911_052334.csv

Results saved to:
  Main results: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection/photon_scan_results_20250911_052334.csv
  Configuration: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection/photon_scan_config_20250911_052334.csv
  Summary statistics: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection/photon_scan_summary_20250911_052334.csv
  All files saved in folder: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection

PHOTON SCAN PARAMETER GRID SCAN SUMMARY
Experiment completed successfully!
  - Generated and analyzed 700000 synthetic images total
  - 7 PFA values tested: [1e-05, 5e-05, 0.0001, 0.0005, 0.001, 0.005, 0.01]
  - 100 pho

## Save Results

In [10]:
folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250910_PhotonScan_SpotDetection/'
file = 'photon_scan_results_20250911_052334.csv'

In [11]:
data = pd.read_csv(os.path.join(folder, file))

In [18]:
xrvals = data.groupby(['method', 'pfa', 'n_photons']).mean().to_xarray()

In [19]:
xrvals

<xarray.Dataset> Size: 186kB
Dimensions:             (method: 3, pfa: 7, n_photons: 100)
Coordinates:
  * method              (method) object 24B 'Raw Bayer' ... 'Variance-Aware'
  * pfa                 (pfa) float64 56B 1e-05 5e-05 0.0001 ... 0.005 0.01
  * n_photons           (n_photons) int64 800B 499 527 556 ... 94788 100000
Data variables:
    image_id            (method, pfa, n_photons) float64 17kB 499.5 ... 499.5
    sensitivity         (method, pfa, n_photons) float64 17kB 0.07647 ... 0.997
    precision           (method, pfa, n_photons) float64 17kB 0.9998 ... 1.0
    f1_score            (method, pfa, n_photons) float64 17kB 0.1418 ... 0.9985
    type_i_error        (method, pfa, n_photons) float64 17kB 0.0001785 ... 0.0
    type_ii_error       (method, pfa, n_photons) float64 17kB 0.9235 ... 0.00...
    true_positives      (method, pfa, n_photons) float64 17kB 27.61 ... 359.9
    false_positives     (method, pfa, n_photons) float64 17kB 0.005 ... 0.0
    false_negatives     (method, pfa, n_photons) float64 17kB 333.4 ... 1.098
    total_ground_truth  (method, pfa, n_photons) float64 17kB 361.0 ... 361.0
    total_detected      (method, pfa, n_photons) float64 17kB 27.61 ... 359.9

In [12]:
import xarray as xr

In [ ]:
type_i_error.method

In [ ]:
fig, axs = plotter.two_column_plot(ncolumns=3, widthratio=[1,1,1])

vmin = np.percentile(type_i_error[:, :, :].to_numpy(), 0)
vmax = np.percentile(type_i_error[:, :, :].to_numpy(), 100)
axs[0] = plotter.image_plot(axs[0], data=type_i_error[0, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,    
                                                        cmap='viridis',

                           cbarlabel='Type-I Error')

axs[1] = plotter.image_plot(axs[1], data=type_i_error[1, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,  
                                                        cmap='viridis',
                            cbar='off',
                           cbarlabel='Type-I Error')

axs[2] = plotter.image_plot(axs[2], data=type_i_error[2, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,
                            cmap='viridis',
                            cbar='off',
                           cbarlabel='Type-I Error')

axs[0].set_title('Raw Image Demosaic')
axs[1].set_title('Photoelectron Demosaic')
axs[2].set_title('Variance Aware Demosaic')

for i in np.arange(3):
    axs[i].set_xticks(np.arange(len(type_i_error.dye)))
    axs[i].set_xticklabels(type_i_error.dye.to_numpy())
    axs[i].tick_params(axis=u'both', which=u'both',length=0)
    
axs[0].set_yticks(np.arange(len(type_i_error.pfa)))
axs[0].set_yticklabels(type_i_error.pfa.to_numpy())
axs[0].tick_params(axis=u'both', which=u'both',length=0)

folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Talks+Posters/Subgroup/20250911/fig/multicolour/spot_detection/'
plt.savefig(os.path.join(folder, 'type_i_error.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
fig, axs = plotter.two_column_plot(ncolumns=3, widthratio=[1,1,1])

vmin = np.percentile(type_ii_error[:, :, :].to_numpy(), 0)
vmax = np.percentile(type_ii_error[:, :, :].to_numpy(), 100)
axs[0] = plotter.image_plot(axs[0], data=type_ii_error[0, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,    
                                                        cmap='viridis',

                           cbarlabel='Type-II Error')

axs[1] = plotter.image_plot(axs[1], data=type_ii_error[1, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,  
                                                        cmap='viridis',
                            cbar='off',
                           cbarlabel='Type-I Error')

axs[2] = plotter.image_plot(axs[2], data=type_ii_error[2, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,
                            cmap='viridis',
                            cbar='off',
                           cbarlabel='Type-I Error')

axs[0].set_title('Raw Image Demosaic')
axs[1].set_title('Photoelectron Demosaic')
axs[2].set_title('Variance Aware Demosaic')

for i in np.arange(3):
    axs[i].set_xticks(np.arange(len(type_ii_error.dye)))
    axs[i].set_xticklabels(type_ii_error.dye.to_numpy())
    axs[i].tick_params(axis=u'both', which=u'both',length=0)
    
axs[0].set_yticks(np.arange(len(type_ii_error.pfa)))
axs[0].set_yticklabels(type_ii_error.pfa.to_numpy())
axs[0].tick_params(axis=u'both', which=u'both',length=0)

folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Talks+Posters/Subgroup/20250911/fig/multicolour/spot_detection/'
plt.savefig(os.path.join(folder, 'type_ii_error.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
fig, axs = plotter.two_column_plot(ncolumns=3, widthratio=[1,1,1])

vmin = np.percentile(sensitivity[:, :, :].to_numpy(), 0)
vmax = np.percentile(sensitivity[:, :, :].to_numpy(), 100)
axs[0] = plotter.image_plot(axs[0], data=sensitivity[0, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,    
                                                        cmap='viridis',

                           cbarlabel='Sensitivity')

axs[1] = plotter.image_plot(axs[1], data=sensitivity[1, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,  
                                                        cmap='viridis',
                            cbar='off',
                           cbarlabel='Type-I Error')

axs[2] = plotter.image_plot(axs[2], data=sensitivity[2, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,
                            cmap='viridis',
                            cbar='off',
                           cbarlabel='Type-I Error')

axs[0].set_title('Raw Image Demosaic')
axs[1].set_title('Photoelectron Demosaic')
axs[2].set_title('Variance Aware Demosaic')

for i in np.arange(3):
    axs[i].set_xticks(np.arange(len(sensitivity.dye)))
    axs[i].set_xticklabels(sensitivity.dye.to_numpy())
    axs[i].tick_params(axis=u'both', which=u'both',length=0)
    
axs[0].set_yticks(np.arange(len(sensitivity.pfa)))
axs[0].set_yticklabels(sensitivity.pfa.to_numpy())
axs[0].tick_params(axis=u'both', which=u'both',length=0)

folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Talks+Posters/Subgroup/20250911/fig/multicolour/spot_detection/'
plt.savefig(os.path.join(folder, 'sensitivity.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
fig, axs = plotter.two_column_plot(ncolumns=3, widthratio=[1,1,1])

vmin = np.percentile(precision[:, :, :].to_numpy(), 0)
vmax = np.percentile(precision[:, :, :].to_numpy(), 100)
axs[0] = plotter.image_plot(axs[0], data=precision[0, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,    
                                                        cmap='viridis',

                           cbarlabel='Sensitivity')

axs[1] = plotter.image_plot(axs[1], data=precision[1, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,  
                                                        cmap='viridis',
                            cbar='off',
                           cbarlabel='Type-I Error')

axs[2] = plotter.image_plot(axs[2], data=precision[2, :, :].to_numpy(), 
                            scalebarlabel='', scalebarsize=0,
                            vmin=vmin,
                            vmax=vmax,
                            cmap='viridis',
                            cbar='off',
                           cbarlabel='Type-I Error')

axs[0].set_title('Raw Image Demosaic')
axs[1].set_title('Photoelectron Demosaic')
axs[2].set_title('Variance Aware Demosaic')

for i in np.arange(3):
    axs[i].set_xticks(np.arange(len(precision.dye)))
    axs[i].set_xticklabels(precision.dye.to_numpy())
    axs[i].tick_params(axis=u'both', which=u'both',length=0)
    
axs[0].set_yticks(np.arange(len(precision.pfa)))
axs[0].set_yticklabels(precision.pfa.to_numpy())
axs[0].tick_params(axis=u'both', which=u'both',length=0)

folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Talks+Posters/Subgroup/20250911/fig/multicolour/spot_detection/'
plt.savefig(os.path.join(folder, 'precision.svg'), dpi=600, format='svg')
plt.show()